# Load input data

In [1]:
from prototype_updated_phase2 import get_experiment_data

X_df5, meta_df5, batch_df5, _ = get_experiment_data(
    design_id="df5" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df5
Design name             : subset_genus_norm
Description             : Balanced subset HIVRC, genus-level, normalized
Aggregation             : genus
Normalize               : True
Cleanset Filtering      : True
Subset studies          : ['Mutlu', 'Dillon', 'Dinh', 'Dubourg', 'Nowak2017']
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (243, 514)
meta_data               : (243, 11)
n_batches               : 4


# Trial investigation

In [2]:
import pandas as pd
import glob

# df6 CSV 파일 찾기
df5_p1_res = pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df5/phase1/optuna_trials_2026-05-20.csv")
print(f"전체 trial 수: {len(df5_p1_res)}")

# 필터링
df_clean = df5_p1_res[
    (df5_p1_res['permanova'] > 0.01) &
    (df5_p1_res['permanova'] < 0.1) &
    (df5_p1_res['noise_ratio'] < 0.65) &
    (df5_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False])

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']].head(20))

전체 trial 수: 1524
의미있는 trial 수: 0
Empty DataFrame
Columns: [trial_number, permanova, n_clusters, noise_ratio, cutoff]
Index: []


In [3]:
import optuna

study = optuna.load_study(
    study_name="latentgee_optuna_df5",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
)

# 상태별 trial 수 확인
from collections import Counter
state_counts = Counter(str(t.state) for t in study.trials)
for state, count in state_counts.items():
    print(f"{state}: {count}")

TrialState.COMPLETE: 1999
TrialState.PRUNED: 1


In [4]:
for study_name in ["latentgee_optuna_df1", "latentgee_optuna_df3", 
                    "latentgee_optuna_df4", "latentgee_optuna_df5", "latentgee_optuna_df6"]:
    study = optuna.load_study(
        study_name=study_name,
        storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
    )
    state_counts = Counter(str(t.state) for t in study.trials)
    print(f"{study_name}: COMPLETE={state_counts.get('TrialState.COMPLETE', 0)}, "
          f"PRUNED={state_counts.get('TrialState.PRUNED', 0)}")

latentgee_optuna_df1: COMPLETE=2000, PRUNED=0
latentgee_optuna_df3: COMPLETE=2000, PRUNED=0
latentgee_optuna_df4: COMPLETE=1892, PRUNED=108
latentgee_optuna_df5: COMPLETE=1999, PRUNED=1
latentgee_optuna_df6: COMPLETE=1889, PRUNED=111


In [5]:
# n_clusters 기준 완화
for min_k in [10, 8, 5, 3]:
    filtered = df5_p1_res[
        (df5_p1_res['permanova'] > 0.01) &
        (df5_p1_res['permanova'] < 0.1) &
        (df5_p1_res['noise_ratio'] < 0.65) &
        (df5_p1_res['n_clusters'] >= min_k)
    ]
    print(f"n_clusters >={min_k}: {len(filtered)}개")


# 조건 완화한 best trial 확인
df_clean2 = df5_p1_res[
        (df5_p1_res['permanova'] > 0.01) &
        (df5_p1_res['permanova'] < 0.1) &
        (df5_p1_res['noise_ratio'] < 0.65)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"\n완화된 필터 결과: {len(df_clean2)}개")
print(df_clean2[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

n_clusters >=10: 0개
n_clusters >=8: 0개
n_clusters >=5: 27개
n_clusters >=3: 234개

완화된 필터 결과: 20개
      trial_number  permanova  n_clusters  noise_ratio  cutoff
775           1054   0.063267           2     0.004115   0.100
619            854   0.098110           2     0.061728   0.100
778           1057   0.050128           2     0.098765   0.100
1454          1914   0.067340           2     0.144033   0.100
360            504   0.078826           2     0.144033   0.100
253            360   0.077020           3     0.156379   0.050
750           1023   0.084594           2     0.156379   0.100
361            507   0.050729           4     0.193416   0.100
1229          1615   0.068440           2     0.201646   0.050
251            359   0.064269           4     0.209877   0.050
654            899   0.032497           3     0.213992   0.070
1446          1908   0.084756           2     0.213992   0.100
1358          1801   0.020657           3     0.234568   0.100
1511          1987   0

# Best trial selection (trial 1054, 358)

In [8]:
print(df5_p1_res[df5_p1_res['trial_number'] == 1054].T)

                               775
cutoff                         0.1
trial_number                  1054
base_dim                       768
n_layers                         4
latent_dim                      56
activation                    gelu
strategy                  constant
dropout_rate                   0.5
epochs                         150
learning_rate             0.000371
loss                      2.121143
permanova                 0.063267
n_clusters                       2
noise_ratio               0.004115
min_cluster_size                 4
min_samples_token              5.0
batch_size                     128
init               kaiming_uniform
beta_kl                   0.069816
weight_decay              0.000006
grad_clip_norm            1.822608
csm                           leaf
kl_warmup_ratio           0.285612
norm                     batchnorm
scheduler                     none


# Run phase 2

In [9]:
from prototype_updated_phase2 import main
main(
    experiment_name="df5",
    phase=2,
    best_trial_number=1054,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df5/phase1/optuna_trials_2026-05-20.csv"
)

main(
    experiment_name="df5",
    phase=2,
    best_trial_number=358,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df5/phase1/optuna_trials_2026-05-20.csv"
)

2026-05-27 10:10:31 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df5/phase2/latentgee_prototype_cutoff_df5_pid1375728_2026-05-27.log
2026-05-27 10:10:31 | LatentGEE 시작 | experiment=df5 | phase=2
2026-05-27 10:10:31 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df5/phase2/config_used.yaml
2026-05-27 10:10:31 | python == 3.10.20
2026-05-27 10:10:31 | torch == 2.2.2
2026-05-27 10:10:31 | numpy == 1.26.4
2026-05-27 10:10:31 | scikit-learn == 1.7.2
2026-05-27 10:10:31 | optuna == 3.6.1
2026-05-27 10:10:31 | hdbscan == 0.8.41
Design ID               : df5
Design name             : subset_genus_norm
Description             : Balanced subset HIVRC, genus-level, normalized
Aggregation             : genus
Normalize               : True
Cleanset Filtering      : True
Subset studies          : ['Mutlu', 'Dillon', 'Dinh', 'Dubourg', 'Nowak2017']
OTU zero-prev           : None
Sample zero-prev        : None
-----------------------------------------

In [12]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.1
X_df5_filtered, meta_df5_filtered, batch_df5_filtered = zero_filter(X_df5, meta_df5, best_cutoff)
X_df5_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df5_filtered_cutoff0.1.csv", index=True)
meta_df5_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df5_filtered_cutoff0.1.csv", index=True)
batch_df5_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df5_filtered_cutoff0.1.csv", index=True)



In [13]:
import numpy as np
import pandas as pd
# X_corrected 로드
X_corrected_df5 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df5/phase2/df5_X_corrected_trial1054_cutoff0.1_2026-05-27.csv",
    index_col=0
)
# inf, NaN 처리
X_corrected_df5_clean = X_corrected_df5.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df5_clean.sum(axis=1).replace(0, 1)
X_corrected_df5_clean = X_corrected_df5_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df5_clean.shape}")
print(f"NaN 수: {X_corrected_df5_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df5_clean.values).sum()}")

X_corrected_df5_clean.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df5/phase2/X_corrected_df5_trial1054_clean_latentgee.csv", index=True)

shape: (243, 155)
NaN 수: 0
inf 수: 0


In [15]:
from functions import evaluate_batch_correction
df5_result = evaluate_batch_correction(
    X_before     = X_df5_filtered,
    X_after      = X_corrected_df5_clean,
    batch_labels = batch_df5_filtered,
    bio_labels   = meta_df5_filtered['hivstatus'],
    renormalize  = True,
    label        = "df5 — LatentGEE (trial 1054)"
)



  df5 — LatentGEE (trial 1054)
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2271  0.0272 -0.1999
PERMANOVA R² (bio) ↑    0.0041  0.0015 -0.0027
kBET acceptance rate ↑  0.0535  0.8971  0.8436
ASW (batch) → 0         0.1005 -0.0225 -0.1230
ASW (bio) ↑            -0.0079 -0.0062  0.0017



In [16]:
best_cutoff = 0.05
X_df5_filtered, meta_df5_filtered, batch_df5_filtered = zero_filter(X_df5, meta_df5, best_cutoff)

X_df5_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df5_filtered_cutoff0.05.csv", index=True)
meta_df5_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df5_filtered_cutoff0.05.csv", index=True)
batch_df5_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df5_filtered_cutoff0.05.csv", index=True)

import numpy as np
import pandas as pd
# X_corrected 로드
X_corrected_df5 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df5/phase2/df5_X_corrected_trial358_cutoff0.05_2026-05-27.csv",
    index_col=0
)
# inf, NaN 처리
X_corrected_df5_clean = X_corrected_df5.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df5_clean.sum(axis=1).replace(0, 1)
X_corrected_df5_clean = X_corrected_df5_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df5_clean.shape}")
print(f"NaN 수: {X_corrected_df5_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df5_clean.values).sum()}")

X_corrected_df5_clean.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df5/phase2/X_corrected_df5_trial358_clean_latentgee.csv", index=True)

shape: (243, 197)
NaN 수: 0
inf 수: 0


# Trial 358을 df5 best trial로 확정

In [17]:
df5_result = evaluate_batch_correction(
    X_before     = X_df5_filtered,
    X_after      = X_corrected_df5_clean,
    batch_labels = batch_df5_filtered,
    bio_labels   = meta_df5_filtered['hivstatus'],
    renormalize  = True,
    label        = "df5 — LatentGEE (trial 358)"
)



  df5 — LatentGEE (trial 358)
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2269  0.0091 -0.2178
PERMANOVA R² (bio) ↑    0.0041  0.0055  0.0014
kBET acceptance rate ↑  0.0535  1.0000  0.9465
ASW (batch) → 0         0.1003 -0.0267 -0.1271
ASW (bio) ↑            -0.0079  0.0011  0.0090

